# CoT Synthesis - FULL RUN (Kaggle GPU / vLLM)

Teacher distillation skala penuh pakai **vLLM lokal** (tanpa limit API).

```
merged_dataset*.jsonl
   -> dedup (merge_and_dedup, by soal)              data/train_pool.jsonl
   -> generate  (teacher vLLM, N solusi/soal)       cot/candidates.jsonl
   -> filter    (judge vLLM) + emit ChatML LANGSUNG sft/{cot,nocot}.jsonl
```

Filter sekarang langsung menghasilkan ChatML (1 solusi terbaik/soal, reasoning Indonesia saja).

**Prasyarat Kaggle:**
1. Settings -> Accelerator = **GPU T4 x2**.
2. Settings -> Internet = **ON** (buat clone repo + download model HF).
3. Add-ons -> Secrets -> tambah `HF_TOKEN` (gemma model gated, butuh login HF).
4. Upload `merged_dataset.jsonl` (dan/atau `merged_dataset_un_cleanvlm_aimo5000.jsonl`) sebagai **Kaggle Dataset**, lalu Add ke notebook.

Full 25k makan waktu lama. Buat coba dulu set `LIMIT` kecil. Buat full: `LIMIT=None`, dan kalau sesi habis, jalankan lagi (resumable lewat `candidates.jsonl`).

## 0. Setup (install + clone repo)

In [ ]:
!pip install -q vllm math-verify
# T4 (sm_75): FlashInfer attention bisa di-compile tapi GAGAL saat jalan ("BatchPrefill invalid argument").
# Buang flashinfer -> vLLM auto fallback ke Triton attention (jalan di T4, tanpa JIT/libcuda).
!pip uninstall -y -q flashinfer-python flashinfer 2>/dev/null; echo "flashinfer dibuang (pakai Triton attention)"

_Catatan: flashinfer sudah dibuang di cell install, jadi vLLM pakai Triton attention — tidak ada JIT FlashInfer, jadi fix libcuda tidak diperlukan lagi. (Kalau suatu saat pakai flashinfer di GPU lain, baru perlu symlink `libcuda.so`.)_

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = '/kaggle/working/FP_NLP'
URL  = 'https://github.com/henray404/FP_NLP.git'  # private -> kalau gagal, lihat cell catatan di bawah

if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)   # selalu ambil kode terbaru
else:
    subprocess.run(['git', 'clone', '-q', URL, REPO], check=True)

sys.path.insert(0, REPO)
WORK = Path('/kaggle/working')
print('repo siap (terbaru):', REPO)
print('CATATAN: kalau ubah src/, restart kernel setelah pull supaya modul lama tidak nyangkut di memori.')

In [ ]:
# HF login via Kaggle Secret (nama secret HARUS persis "HF_TOKEN").
# Add-ons -> Secrets -> tambah HF_TOKEN, lalu attach ke notebook ini.
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

tok = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = tok
os.environ["HUGGING_FACE_HUB_TOKEN"] = tok   # vLLM/transformers baca var ini juga
login(token=tok)
print("HF login OK")

## 1. Cari data (Kaggle Dataset yang di-Add)

In [ ]:
INPUTS = [str(p) for p in Path('/kaggle/input').rglob('merged_dataset*.jsonl')]
assert INPUTS, 'Tidak ketemu. Upload merged_dataset*.jsonl sebagai Kaggle Dataset lalu Add ke notebook.'
print('input files:')
for p in INPUTS:
    print(' -', p)

## 2. Dedup + buang soal tanpa jawaban

Dedup by `soal` (simpan versi paling lengkap). Lalu buang baris tanpa `jawaban` (judge butuh gold answer).

In [ ]:
import json
from src.preprop.merge_and_dedup import run as merge_dedup

DEDUP = WORK / 'dataset_dedup.jsonl'
print(merge_dedup([Path(p) for p in INPUTS], DEDUP))

POOL = WORK / 'train_pool.jsonl'
n = 0
with open(DEDUP, encoding='utf-8') as f, open(POOL, 'w', encoding='utf-8') as o:
    for line in f:
        d = json.loads(line)
        if (d.get('jawaban') or '').strip():
            o.write(json.dumps(d, ensure_ascii=False) + '\n')
            n += 1
print('train_pool (ada jawaban):', n, '->', POOL)

## 3. Config

In [ ]:
# Notebook ini buat PERBANDINGAN generator: teacher = Gemma-2 (beda dari cot_pipeline_kaggle yg
# pakai DeepSeek-R1). vLLM gak bisa jalanin Gemma-2 di T4 (head_dim=256 -> Triton OutOfResources,
# V0 engine dihapus), jadi generate pakai backend 'hf' (transformers lokal) -> tetap no-API/no-limit.
# Judge tetap vLLM (Qwen head_dim=128, aman di T4).
TEACHER_MODEL = 'google/gemma-2-2b-it'   # teacher generator (yg dibandingkan)
JUDGE_MODEL   = 'Qwen/Qwen2.5-7B-Instruct'                # judge

N_CANDIDATES = 4       # solusi per soal
MAX_TOKENS   = 4096    # reasoning panjang; lokal jadi gak ada limit TPM
TP           = 2       # 2x T4 -> tensor parallel 2 (dipakai judge/filter vLLM)
LIMIT        = 50      # COBA dulu. Full: None

# Solo full run: NUM_SHARDS=1, SHARD_I=0 (proses semua soal, no split).
# Paralel berdua: kamu SHARD_I=0, teman SHARD_I=1, dua-duanya NUM_SHARDS=2 -> concat candidates.jsonl.
NUM_SHARDS = 1
SHARD_I    = 0

COT_DIR = WORK / 'cot'
SFT_DIR = WORK / 'sft'
CAND    = COT_DIR / 'candidates.jsonl'
CORRECT = COT_DIR / 'correct.jsonl'
print('teacher:', TEACHER_MODEL, '| N:', N_CANDIDATES, '| limit:', LIMIT,
      '| shard:', f'{SHARD_I}/{NUM_SHARDS}')

## 3b. Resume antar-session (checkpoint)

Biar bisa **stop session** tanpa ngulang dari 0:

1. **Cara paling gampang** — Notebook *Settings* -> **Persistence** -> `Files only` (atau `Variables and Files`).
`/kaggle/working` selamat waktu stop, jadi `candidates.jsonl` tetap ada -> run berikut auto-lanjut.
2. **Backup** (kalau persistence off / ganti notebook): zip output (cell terakhir) -> download ->
upload sebagai Kaggle Dataset -> **Add** ke notebook. Cell di bawah otomatis pulihkan dari `/kaggle/input`.

Yang lanjut otomatis: generate skip soal yang sudah `N` kandidat; filter skip kandidat yang sudah dinilai (`correct.jsonl.progress`).

In [ ]:
import shutil
COT_DIR.mkdir(parents=True, exist_ok=True)

# Pulihkan checkpoint dari Dataset yang di-Add (kalau /kaggle/working ke-reset).
for fname in ['candidates.jsonl', 'correct.jsonl', 'correct.jsonl.progress']:
    dst = COT_DIR / fname
    if dst.exists():
        continue
    hits = list(Path('/kaggle/input').rglob(fname))
    if hits:
        shutil.copy(hits[0], dst)
        print('restore', fname, '<-', hits[0])

# Status checkpoint
if CAND.exists():
    n = sum(1 for _ in open(CAND, encoding='utf-8'))
    print(f'candidates.jsonl: {n} baris kandidat -> generate lanjut dari sini')
else:
    print('belum ada candidates.jsonl -> generate mulai dari 0')
if Path(str(CORRECT) + '.progress').exists():
    n = sum(1 for _ in open(str(CORRECT) + '.progress', encoding='utf-8'))
    print(f'progress judge: {n} kandidat sudah dinilai -> filter skip yang ini')

## 4. Generate (teacher Gemma-2, transformers lokal)

Notebook ini bandingin generator **Gemma-2** (vs DeepSeek-R1 di `cot_pipeline_kaggle.ipynb`).

**Kenapa transformers, bukan vLLM (dan bukan API):** vLLM gak bisa jalanin Gemma-2 di T4 — `head_dim=256`
bikin kernel Triton attention minta shared memory > limit T4 (sm75 64KB) -> `OutOfResources` /
`EngineDeadError`, dan vLLM baru sudah buang engine V0 jadi workaround env diabaikan. Backend `hf`
(transformers, `attn_implementation="eager"`) jalan lokal tanpa limit itu — **bukan API, no rate
limit**. Gemma-2-2b (2B) muat santai di 1x T4. Lebih lambat dari vLLM tapi stabil; tulis incremental
jadi aman ke-timeout. Resumable: jalankan lagi, soal yang sudah `N` kandidat dilewati.

In [ ]:
import subprocess
# libcuda symlink (jaga-jaga buat lib lain). Generate pakai backend transformers, jadi tidak
# butuh env attention vLLM apa pun.
libs = subprocess.run("find / -name 'libcuda.so*' 2>/dev/null",
                    shell=True, capture_output=True, text=True).stdout.split()
src = next((l for l in libs if l.endswith('libcuda.so.1')), libs[0] if libs else '')
for d in ['/usr/local/cuda/lib64', '/usr/local/cuda/lib64/stubs']:
    subprocess.run(f'mkdir -p {d} && ln -sf {src} {d}/libcuda.so', shell=True)
subprocess.run('echo /usr/local/cuda/lib64 > /etc/ld.so.conf.d/zzz_cuda.conf && ldconfig',
shell=True)
print('libcuda.so ->', src)

In [ ]:
from src.cot_synthesis.generate_gemma import run_generate

# backend='hf': transformers lokal (eager attention). Bukan API -> no rate limit, full lokal.
# vLLM gak dipakai DI SINI karena crash buat Gemma-2 di T4 (head_dim=256 Triton OutOfResources).
# Gemma-2-2b cuma 2B -> muat 1x T4. Batched + resumable: tulis incremental per soal.
gen_stats = run_generate(
    POOL, CAND, backend='hf', model=TEACHER_MODEL,
    n=N_CANDIDATES, max_tokens=MAX_TOKENS, limit=LIMIT,
    shard=SHARD_I, num_shards=NUM_SHARDS,
)
print(gen_stats)

## 5. Bebaskan GPU sebelum load judge

Kalau cell filter di bawah **OOM**: Restart kernel, lalu jalankan ulang cell Setup -> Config -> Filter (file `candidates.jsonl` di /kaggle/working tetap ada).

In [ ]:
from src.cot_synthesis.filter_solutions import run_filter

# Filter pakai LLM judge, lalu LANGSUNG bangun ChatML (cot.jsonl + nocot.jsonl) -> pipeline
# berakhir di format training. best_per_problem=True: 1 solusi terbaik/soal (Indonesia + terpendek).
# id_only=True: buang reasoning Inggris dari arm CoT. Set keep_all_per_problem via best_per_problem=False.
flt_stats = run_filter(
    CAND, CORRECT, judge_backend='vllm', judge_model=JUDGE_MODEL, tensor_parallel_size=TP,
    emit_chatml=True, chatml_dir=SFT_DIR, best_per_problem=True, id_only=True,
)
acc = flt_stats['kept'] / max(flt_stats['total'], 1)
print(flt_stats)
print(f'acceptance rate: {acc:.1%}')

## 7. (Opsional) Bangun ulang ChatML

Cell Filter di atas **sudah** menghasilkan `sft/cot.jsonl` + `nocot.jsonl` (emit_chatml=True).
Jalankan cell ini hanya kalau mau ganti policy (mis. simpan semua solusi/soal, atau izinkan Inggris).

In [ ]:
# Opsional: bangun ulang ChatML dari correct.jsonl dengan policy berbeda.
from src.cot_synthesis.to_chatml import run as build_chatml

ml_stats = build_chatml(CORRECT, SFT_DIR, best_per_problem=True, id_only=True,
                        max_solution_chars=8000)
print(ml_stats)

## 8. Download hasil

Output ada di `/kaggle/working/sft/` dan `/kaggle/working/cot/`. Zip biar gampang diunduh, atau pakai **Save Version** biar persist antar sesi.

In [ ]:
import shutil
from IPython.display import FileLink

out_zip = '/kaggle/working/cot_sft_outputs'
# kumpulin cot/ + sft/ ke satu folder lalu zip
bundle = WORK / 'bundle'
bundle.mkdir(exist_ok=True)
for d in ['cot', 'sft']:
    src = WORK / d
    if src.exists():
        shutil.copytree(src, bundle / d, dirs_exist_ok=True)
shutil.make_archive(out_zip, 'zip', str(bundle))
print('zip:', out_zip + '.zip')
FileLink('cot_sft_outputs.zip')